##### Split
- split() is used in PySpark to **divide a string column** into an **array** column based on a **delimiter or regex**.
- It Convert **String** Column to **Array**.
- Returns an **array type** after splitting the **string column** by **delimiter**.
- The split function takes two arguments: the name of the **column** to split and the **delimiter**.

**When to use split() vs explode()**

| Function    | Purpose                   |
| ----------- | ------------------------- |
| `split()`   | Converts `string → array` |
| `explode()` | Converts `array → rows`   |

**Topics Covered**
- How to split `string`?
- Split on a `string column` using `space as delimiter`
- Split() on a `string column` using `comma as delimiter`
- Split using `regex (multiple delimiters)`

In [0]:
from pyspark.sql.types import IntegerType, StringType, ArrayType, StructType, StructField
from pyspark.sql.functions import split, array

##### 1) How to split string?

In [0]:
primary_keys= "SALES_ID|EVENT_ID|TASK_ID|PURCHASE_ID|CUSTOM_ID"
print("input: ", primary_keys)
print("data type: ", type(primary_keys))

lstpKeys = primary_keys.split("|")
print("\noutput: ", lstpKeys)
print("data type: ", type(lstpKeys))

input:  SALES_ID|EVENT_ID|TASK_ID|PURCHASE_ID|CUSTOM_ID
data type:  <class 'str'>

output:  ['SALES_ID', 'EVENT_ID', 'TASK_ID', 'PURCHASE_ID', 'CUSTOM_ID']
data type:  <class 'list'>


In [0]:
string = "Azure, DataBricks, Spark, ML"
print("input: ", string)
print("data type: ", type(string))


array = string.split(",")
print("\noutput: ", array)
print("data type: ", type(array))

input:  Azure, DataBricks, Spark, ML
data type:  <class 'str'>

output:  ['Azure', ' DataBricks', ' Spark', ' ML']
data type:  <class 'list'>


In [0]:
string = "Azure Data Engineer"
print("input: ", string)
print("data type: ", type(string))
      
array = string.split(" ")
print("\noutput: ", array)
print("data type: ", type(array))

input:  Azure Data Engineer
data type:  <class 'str'>

output:  ['Azure', 'Data', 'Engineer']
data type:  <class 'list'>


##### 2) Split on a `string column` using `space as delimiter`

In [0]:
data = [("Big data processing",)]
df1 = spark.createDataFrame(data, ["sentence"])
display(df1)

df1.withColumn("words", split("sentence", " ")).display()

sentence
Big data processing


sentence,words
Big data processing,"List(Big, data, processing)"


##### 3) split() on a `string column` using `comma as delimiter`

In [0]:
data = [("red,blue,green",)]
df = spark.createDataFrame(data, ["colors"])
display(df)

colors
"red,blue,green"


In [0]:
df.withColumn("colors_array", split("colors", ",")).display()

colors,colors_array
"red,blue,green","List(red, blue, green)"


In [0]:
# Input data
input_data = [
    ("Washing machine", "red,blue,green", 1000),
    ("Vaccum cleaner", "grey,white", 200),
    ("Iron", "black,white,", 50),
    ("Blender", "red,,None", 30),
    ("Hair dryer", "red,,", 20),
    ("Hair straightner", ",,,", 20),
    ("Cleaning spray", "red,,None", 10),
    ("Hair spray", "", 10)
]

input_df = spark.createDataFrame(input_data, ["Material", "color", "Price"])

display(input_df)

Material,color,Price
Washing machine,"red,blue,green",1000
Vaccum cleaner,"grey,white",200
Iron,"black,white,",50
Blender,"red,,None",30
Hair dryer,"red,,",20
Hair straightner,",,,",20
Cleaning spray,"red,,None",10
Hair spray,,10


In [0]:
# split() → breaks "red,blue,green" into a list of colors
# explode() → explodes the list of colors into separate rows
# collect_list() → collects the colors into a list

output_df = input_df.withColumn("color_split", split("color", ","))
display(output_df)

Material,color,Price,color_split
Washing machine,"red,blue,green",1000,"List(red, blue, green)"
Vaccum cleaner,"grey,white",200,"List(grey, white)"
Iron,"black,white,",50,"List(black, white, )"
Blender,"red,,None",30,"List(red, , None)"
Hair dryer,"red,,",20,"List(red, , )"
Hair straightner,",,,",20,"List(, , , )"
Cleaning spray,"red,,None",10,"List(red, , None)"
Hair spray,,10,List()


**Use Case: 02**

In [0]:
data = [("Amar,,Singh",["Java","Scala","C++"], ["Spark","Java","Azure Databricks"], [8, 9, 5, 7], "Bangalore", "Chennai", 25, 7),
        ("Ramesh,Rathode,", ["Python","PySpark","C"], ["spark sql","ADF"], [11, 3, 6, 8], "Hyderabad", "Kochin", 35, 8),
        ("Asha,,Rani", ["Devops","VB","Git"], ["ApacheSpark","Python"], [5, 6, 8, 10], "Amaravathi", "Noida", 30, 10),
        ("Rakesh,Kothur,", ["SQL","Azure","AWS"], ["PySpark","Oracle","Confluence"], [12, 6, 8, 15], "Noida", "Mumbai", 33, 5),
        ("Krishna,,Joshi", ["GCC","Visual Studio"], ["SQL","Databricks","SQL Editor"], [2, 6, 5, 8], "Delhi", "Kolkata", 28, 6),
        ("Hari,,Rani", ["Devops","VB","Git"], ["ApacheSpark","Python"], [5, 6, 8, 10], "Amaravathi", "Noida", 30, 10),
        ("Rakesh,kumar,", ["SQL","Azure","AWS"], ["PySpark","Oracle","Schema"], [12, 6, 8, 15], "luknow", "Mumbai", 33, 5),
        ("karan,,Joshi", ["AWS","Visual Studio"], ["SQL","Git","SQL Editor"], [2, 6, 5, 8], "Delhi", "Noida", 28, 6),
        ]

schema = StructType([ 
    StructField("FullName", StringType(), True), 
    StructField("LearntLanguages", ArrayType(StringType()), True), 
    StructField("ToLearnLanguages", ArrayType(StringType()), True),
    StructField("Rating", ArrayType(IntegerType()), True), 
    StructField("PresentState", StringType(), True), 
    StructField("PreviousState", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Experience", IntegerType(), True)
  ])

df = spark.createDataFrame(data=data, schema=schema)
df.printSchema()
display(df)

root
 |-- FullName: string (nullable = true)
 |-- LearntLanguages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ToLearnLanguages: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- Rating: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- PresentState: string (nullable = true)
 |-- PreviousState: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)



FullName,LearntLanguages,ToLearnLanguages,Rating,PresentState,PreviousState,Age,Experience
"Amar,,Singh","List(Java, Scala, C++)","List(Spark, Java, Azure Databricks)","List(8, 9, 5, 7)",Bangalore,Chennai,25,7
"Ramesh,Rathode,","List(Python, PySpark, C)","List(spark sql, ADF)","List(11, 3, 6, 8)",Hyderabad,Kochin,35,8
"Asha,,Rani","List(Devops, VB, Git)","List(ApacheSpark, Python)","List(5, 6, 8, 10)",Amaravathi,Noida,30,10
"Rakesh,Kothur,","List(SQL, Azure, AWS)","List(PySpark, Oracle, Confluence)","List(12, 6, 8, 15)",Noida,Mumbai,33,5
"Krishna,,Joshi","List(GCC, Visual Studio)","List(SQL, Databricks, SQL Editor)","List(2, 6, 5, 8)",Delhi,Kolkata,28,6
"Hari,,Rani","List(Devops, VB, Git)","List(ApacheSpark, Python)","List(5, 6, 8, 10)",Amaravathi,Noida,30,10
"Rakesh,kumar,","List(SQL, Azure, AWS)","List(PySpark, Oracle, Schema)","List(12, 6, 8, 15)",luknow,Mumbai,33,5
"karan,,Joshi","List(AWS, Visual Studio)","List(SQL, Git, SQL Editor)","List(2, 6, 5, 8)",Delhi,Noida,28,6


In [0]:
# converts the string column into an array of strings
df_spl = df.select("FullName", split("FullName",",").alias("FullNameSplit"),
                               split("PresentState",",").alias("PresentStateSplit"),
                               split("PreviousState",",").alias("PreviousStateSplit"))
display(df_spl)

FullName,FullNameSplit,PresentStateSplit,PreviousStateSplit
"Amar,,Singh","List(Amar, , Singh)",List(Bangalore),List(Chennai)
"Ramesh,Rathode,","List(Ramesh, Rathode, )",List(Hyderabad),List(Kochin)
"Asha,,Rani","List(Asha, , Rani)",List(Amaravathi),List(Noida)
"Rakesh,Kothur,","List(Rakesh, Kothur, )",List(Noida),List(Mumbai)
"Krishna,,Joshi","List(Krishna, , Joshi)",List(Delhi),List(Kolkata)
"Hari,,Rani","List(Hari, , Rani)",List(Amaravathi),List(Noida)
"Rakesh,kumar,","List(Rakesh, kumar, )",List(luknow),List(Mumbai)
"karan,,Joshi","List(karan, , Joshi)",List(Delhi),List(Noida)


##### 4) Split using regex (multiple delimiters)

In [0]:
data = [("red|blue,green",)]
df2 = spark.createDataFrame(data, ["colors"])
display(df2)

df2.withColumn("colors_array", split("colors", "[|,]")).display(df2)

colors
"red|blue,green"


colors,colors_array
"red|blue,green","List(red, blue, green)"
